# PSI-159 / CFI-G — model 3D płomienia z **INTNS** (właściwe źródło) + walidacja wideo

**Dlaczego INTNS, a nie HOBJ/ACME:** kamera ACME (`Images.zip`, `.hobj`) to kamera widzialna —
pokazuje głównie statyczne struktury komory i żarzenie ignitera. Płomień widać dopiero na kamerze
**INTNS** (wzmocnionej): klatki `INTNS_21088A1_*.tiff`, 1392×1040, **Mono16** (16-bit, ~23,22 fps,
2017 klatek, ekspozycja 43067/32000, gain 143).

**Pipeline:** TIFF → tło z klatek przed zapłonem → oś symetrii (maks. symetrii lewo/prawo) →
profil radialny → **inwersja Abela** → ε(r,z) 256×128 → `lut.bin` + `index.json`.

**Walidacja:** klatki referencyjne z nagrania wideo `.ts` (640×480, 29,97 fps) — płomień jest
kompaktowy, lekko pusty w środku (ring test: środek ~20% ciemniejszy od pierścienia), od zapłonu
~137 s do ~186 s nagrania.

In [ ]:
#@title 1. Zależności
import os, sys, io, json, zipfile, hashlib, re, subprocess
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    !apt-get -qq install -y ffmpeg >/dev/null 2>&1 || true

In [ ]:
#@title 2. Konfiguracja
TEST_ID   = '21088A1'
INTNS_ZIP = 'PSI-159_Raw Data_21088A1_INTNS_ALL.zip'      # 2,28 GB
VIDEO     = 'PSI-159_Raw Data_vlc-record-2021-03-29-15h24m41s-udp____21088A1.ts'  # 490 MB
IGN_FRAME, EXT_FRAME = 468, 1814      # z TP.txt: zapłon 15:27:13, wygaśnięcie 15:28:11 (~23.22 fps)
OUT_DIR = f'/content/psi159_intns/{TEST_ID}'
os.makedirs(OUT_DIR, exist_ok=True)

def fetch_psi(name, dest):
    """Pobiera plik z NASA przez krótkotrwały presigned URL."""
    url = subprocess.check_output(['curl','-fsSG',
        '--data-urlencode',f'file={name}','--data-urlencode','version=1',
        '--data-urlencode','redirect=false',
        'https://psi.nasa.gov/geode-py/ws/studies/PSI-159/download'], text=True).strip()
    subprocess.run(['curl','-fL','--retry','3',url,'-o',dest], check=True)
    print(name, '->', dest, os.path.getsize(dest), 'B')

In [ ]:
#@title 3. Pobranie INTNS + wideo (jeśli brak)
intns_path = f'/content/{INTNS_ZIP}'
video_path = f'/content/{VIDEO}'
if not os.path.exists(intns_path): fetch_psi(INTNS_ZIP, intns_path)
if not os.path.exists(video_path): fetch_psi(VIDEO, video_path)

In [ ]:
#@title 4. Funkcje: HOBJ/TIFF, filtr Gaussa, Abel
def load_tiff(zf, name):
    return np.asarray(Image.open(io.BytesIO(zf.read(name)))).astype(np.float32)

def gauss(a, s):
    if s <= 0: return a
    k = int(3*s)|1; x = np.arange(k)-k//2
    w = np.exp(-(x*x)/(2*s*s)); w /= w.sum()
    b = np.apply_along_axis(lambda r: np.convolve(r,w,'same'),1,a)
    return np.apply_along_axis(lambda c: np.convolve(c,w,'same'),0,b)

def abel_matrix(N):
    A = np.zeros((N,N))
    for k in range(N):
        for j in range(k,N):
            A[k,j] = 2*(np.sqrt(max((j+1)**2-k*k,0)) - np.sqrt(max(j*j-k*k,0)))
    return A

def abel_invert(profs, A):
    nrows, N = profs.shape
    f = np.zeros_like(profs)
    for k in range(N-1,-1,-1):
        s = f[:,k+1:] @ A[k,k+1:] if k+1 < N else np.zeros(nrows)
        f[:,k] = (profs[:,k]-s)/max(A[k,k],1e-9)
    return np.clip(f,0,None)

In [ ]:
#@title 5. Budowa LUT z INTNS
zf = zipfile.ZipFile(intns_path)
names = sorted(n for n in zf.namelist() if n.endswith('.tiff'))
N = len(names); H, W = 1040, 1392
print('klatek INTNS:', N)
bg = np.median(np.stack([load_tiff(zf,names[i]) for i in range(100,400,20)]), axis=0)

# oś symetrii z klatki mniej więcej w środku spalania
s0 = gauss(np.clip(load_tiff(zf, names[(IGN_FRAME+EXT_FRAME)//2]) - bg, 0, None), 3.0)
def sym(cx):
    t=0.0
    for d in range(1,300):
        if cx-d<0 or cx+d>=W: break
        t += float((s0[:,cx-d]*s0[:,cx+d]).sum())
    return t
cx = max(range(300,W-300,4), key=sym)
print('oś symetrii cx =', cx)

def reconstruct(i):
    s = gauss(np.clip(load_tiff(zf,names[i]) - bg, 0, None), 3.0)
    col = s[:, max(0,cx-350):cx+350].mean(axis=1)
    ys = np.where(col > 0.25*col.max())[0]
    if len(ys) < 10: return None
    cy = int((ys.min()+ys.max())//2); R = int((ys.max()-ys.min())//2)
    R = min(R, cx, W-cx, cy, H-cy)
    if R < 40: return None
    r = np.arange(0,R,3).astype(int); M = len(r); A = abel_matrix(M)
    rows = list(range(cy-R, cy+R, 3))
    profs = np.array([0.5*(s[y,cx-r]+s[y,cx+r]) for y in rows])
    eps = abel_invert(profs, A)
    return np.array(Image.fromarray(eps.astype(np.float32), mode='F').resize((128,256), Image.BILINEAR))

idx = sorted(set(np.linspace(0, N-1, 120).astype(int).tolist()))
raw, meta = [], []
for k,i in enumerate(idx):
    f = load_tiff(zf, names[i])
    Image.fromarray((np.clip(f/6000.0,0,1)*255).astype(np.uint8)).resize((256,192), Image.BILINEAR).save(f'{OUT_DIR}/frames/f{k:03d}.jpg', quality=82)
    meta.append({'i':k,'src':i,'file':f'f{k:03d}.jpg'})
    e = reconstruct(i) if IGN_FRAME <= i <= EXT_FRAME else None
    raw.append(e if e is not None else np.zeros((256,128), np.float32))
gm = max(float(e.max()) for e in raw) or 1.0
lut = np.stack([np.clip(e/gm*255,0,255).astype(np.uint8) for e in raw])
lut.tofile(f'{OUT_DIR}/lut.bin')
json.dump({'dataset':'PSI-159','testpoint':TEST_ID,'camera':'INTNS (intensified, Mono16 TIFF)',
           'sourceFile':INTNS_ZIP,'frames':len(raw),'z':256,'r':128,'axisX':int(cx),
           'ignitionFrame':IGN_FRAME,'extinctionFrame':EXT_FRAME,'fps':12,'globalMax':gm,
           'sha256':hashlib.sha256(open(intns_path,'rb').read()).hexdigest()},
          open(f'{OUT_DIR}/lut-index.json','w'), indent=1)
json.dump({'testpoint':TEST_ID,'totalFrames':N,'w':W,'h':H,'frames':meta},
          open(f'{OUT_DIR}/index.json','w'), indent=1)
print('LUT:', lut.nbytes, 'B, globalMax', round(gm,1))

In [ ]:
#@title 6. Walidacja klatką źródłową (projekcja zwrotna)
k = 60
f = load_tiff(zf, names[idx[k]]); s = gauss(np.clip(f-bg,0,None), 3.0)
fig, ax = plt.subplots(1,2, figsize=(10,4))
ax[0].imshow(s, cmap='inferno', aspect='auto'); ax[0].axvline(cx, color='cyan', lw=0.8)
ax[0].set_title(f'INTNS #{idx[k]} — sygnał')
ax[1].imshow(lut[k], cmap='inferno', aspect='auto'); ax[1].set_title('ε(r,z) — Abel')
plt.tight_layout(); plt.show()

In [ ]:
#@title 7. Walidacja wideo — klatki referencyjne (ffmpeg)
REF = '/content/video_ref'; os.makedirs(REF, exist_ok=True)
# zapłon ~137 s, główne spalanie ~145 s (okno wg analizy nagrania)
for t in [137.07, 145.0, 160.0, 180.0]:
    subprocess.run(['ffmpeg','-y','-ss',str(t),'-i',video_path,'-frames:v','1',
                    f'{REF}/f_{t}.png'], check=True, capture_output=True)
    im = np.asarray(Image.open(f'{REF}/f_{t}.png').convert('RGB')).astype(float)
    L = 0.299*im[...,0]+0.587*im[...,1]+0.114*im[...,2]
    m = L > 0.5*L.max()
    ys, xs = np.nonzero(m)
    print(f't={t}s  mean={L.mean():.1f}  max={L.max():.0f}  glow bbox=({xs.min()},{ys.min()})-({xs.max()},{ys.max()})')
plt.imshow(Image.open(f'{REF}/f_145.0.png')); plt.title('wideo: główne spalanie (145 s)')
plt.axis('off'); plt.show()

In [ ]:
#@title 8. Zapis wyników na Drive
if IN_COLAB:
    import shutil
    dest = f'/content/drive/MyDrive/psi159_intns/{TEST_ID}'
    shutil.copytree(OUT_DIR, dest, dirs_exist_ok=True)
    print('skopiowano do', dest)

## Jak podłączyć do aplikacji
Skopiuj `lut.bin`, `lut-index.json`, `index.json`, `frames/` do `app/app/public/psi159/21088A1/` —
aplikacja (widok **Flame VR**) wczyta je automatycznie.

## Ograniczenia
- Inwersja Abela zakłada **symetrię osiową**; uśrednienie lewo/prawo usuwa rzeczywistą asymetrię.
- To nadal **aproksymacja** (ε(r,z) z jednego rzutu), a nie pomiar wolumenu 3D.
- Wideo jest 8-bitowe, z kompresją H.264 i utratą pakietów UDP — służy tylko jako **referencja
  kształtu**, nie do rekonstrukcji ilościowej.